# ChatPromptTemplate的高级特性

## 1、部分变量预填充：partial()

举例1：

In [1]:
from langchain_core.prompts import ChatPromptTemplate

# 原始模板
template = ChatPromptTemplate.from_messages([
    ("system", "你是{role}，目标用户是{audience}"),
    ("user", "{task}")
])

result1 = template.invoke({"role":"导游", "audience":"游客", "task":"介绍一下北京的故宫"})
result2 = template.invoke({"role":"导游", "audience":"游客", "task":"介绍一下北京的颐和园"})

print(result1)
print(result2)

messages=[SystemMessage(content='你是导游，目标用户是游客', additional_kwargs={}, response_metadata={}), HumanMessage(content='介绍一下北京的故宫', additional_kwargs={}, response_metadata={})]
messages=[SystemMessage(content='你是导游，目标用户是游客', additional_kwargs={}, response_metadata={}), HumanMessage(content='介绍一下北京的颐和园', additional_kwargs={}, response_metadata={})]


上述代码可以使用partial()优化

In [5]:
from langchain_core.prompts import ChatPromptTemplate

# 原始模板
template = ChatPromptTemplate.from_messages([
    ("system", "你是{role}，目标用户是{audience}"),
    ("user", "{task}")
])

# 部分变量预填充
final_template = template.partial(role = "导游", audience = "游客")

result1 = final_template.format_messages(task = "介绍一下北京的故宫")
result2 = final_template.format_messages(task = "介绍一下北京的颐和园")

print(result1)
print(result2)

[SystemMessage(content='你是导游，目标用户是游客', additional_kwargs={}, response_metadata={}), HumanMessage(content='介绍一下北京的故宫', additional_kwargs={}, response_metadata={})]
[SystemMessage(content='你是导游，目标用户是游客', additional_kwargs={}, response_metadata={}), HumanMessage(content='介绍一下北京的颐和园', additional_kwargs={}, response_metadata={})]


举例2：

In [11]:
# 场景：为不同部门创建专用模板
base_template = ChatPromptTemplate.from_messages([
    ("system", "你是{department}的{role}"),
    ("user", "{task}")
])
# IT 部门
it_template = base_template.partial(
    department="IT部门",
    role="技术支持"
)
# 销售部门
sales_template = base_template.partial(
    department="销售部门",
    role="销售顾问"
)

result1 = sales_template.invoke({"task":"为什么每年年底汽车会促销？"})
result2 = it_template.invoke({"task":"每年年底汽车促销的时候是如何解决平台高并发问题？"})

print(result1)
print(result2)

messages=[SystemMessage(content='你是销售部门的销售顾问', additional_kwargs={}, response_metadata={}), HumanMessage(content='为什么每年年底汽车会促销？', additional_kwargs={}, response_metadata={})]
messages=[SystemMessage(content='你是IT部门的技术支持', additional_kwargs={}, response_metadata={}), HumanMessage(content='每年年底汽车促销的时候是如何解决平台高并发问题？', additional_kwargs={}, response_metadata={})]


## 2、消息占位符

### 2.1 使用placeholder

举例：

In [12]:
template = ChatPromptTemplate.from_messages([
    ("system", "我是一个AI助手"),
    ("placeholder", "{conversation}")
])

result = template.invoke({
    "conversation" : [
        ("human", "你好，请问明天的天气如何？"),
        ("ai", "明天天气晴朗"),
        ("human", "你好，请问后天的天气如何？"),
    ]
})
print(result)

messages=[SystemMessage(content='我是一个AI助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好，请问明天的天气如何？', additional_kwargs={}, response_metadata={}), AIMessage(content='明天天气晴朗', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你好，请问后天的天气如何？', additional_kwargs={}, response_metadata={})]


### 2.2 使用MessagesPlaceholder

举例：

In [14]:
from langchain_core.prompts import MessagesPlaceholder
template = ChatPromptTemplate.from_messages([
    ("system", "我是一个AI助手"),
    MessagesPlaceholder(variable_name="conversation")
])

result = template.invoke({
    "conversation" : [
        ("human", "你好，请问明天的天气如何？"),
        ("ai", "明天天气晴朗"),
        ("human", "你好，请问后天的天气如何？"),
    ]
})
print(result)

messages=[SystemMessage(content='我是一个AI助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好，请问明天的天气如何？', additional_kwargs={}, response_metadata={}), AIMessage(content='明天天气晴朗', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你好，请问后天的天气如何？', additional_kwargs={}, response_metadata={})]


In [15]:
from langchain_core.messages import HumanMessage, AIMessage
result = template.invoke({
    "conversation" : [
        HumanMessage("你好，请问明天的天气如何？"),
        AIMessage("明天天气晴朗"),
        HumanMessage("你好，请问后天的天气如何？")
    ]
})
print(result)

messages=[SystemMessage(content='我是一个AI助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好，请问明天的天气如何？', additional_kwargs={}, response_metadata={}), AIMessage(content='明天天气晴朗', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='你好，请问后天的天气如何？', additional_kwargs={}, response_metadata={})]


举例：存储历史消息

In [23]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 1、提供大模型
load_dotenv(override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")
DASHSCOPE_BASE_URL = os.getenv("DASHSCOPE_BASE_URL")

model = init_chat_model(
    model="qwen3.7-plus",
    model_provider="openai",
    # temperature=1.5,
    api_key=DASHSCOPE_API_KEY,
    base_url=DASHSCOPE_BASE_URL,
    # max_tokens=10,
)

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "你是一个非常友好的AI助手"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}")
    ]
)
result1 = prompt_template.invoke(
    {
        "history": [
            ("human", "5 + 2 = ?"),
            ("ai", "5 + 2 = 7")
        ],
        "question": "结果再乘以4呢？"
    }
)

response1 = model.invoke(result)
print(response1)

result2 = prompt_template.invoke(
    {
        "history": [
            ("human", "5 + 2 = ?"),
            ("ai", "5 + 2 = 7") ,
            ("human", "结果再乘以4呢？"),
            ("ai", "7 乘以 4 等于 28。")
        ],
        "question": "问的第一个问题是什么？"
    }
)

response2 = model.invoke(result2)
print(response2)

content='结果再乘以4的话，就是 7 × 4 = 28 啦！如果还有其他问题，随时告诉我哦。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 82, 'prompt_tokens': 49, 'total_tokens': 131, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 50, 'rejected_prediction_tokens': None, 'text_tokens': 82}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0, 'text_tokens': 49}}, 'model_provider': 'openai', 'model_name': 'qwen3.7-plus', 'system_fingerprint': None, 'id': 'chatcmpl-6c03f0eb-c919-9796-bf36-60ab542565bc', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f431a-5ecd-7331-86f4-40aac2db7642-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 49, 'output_tokens': 82, 'total_tokens': 131, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 50}}
content='您问的第一个问题是：“5 + 2 = ?”' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tok

## 3、可复用模板库

举例1：定义了具体的存放模板的py文件

In [25]:
from langchain_core.prompts import ChatPromptTemplate

class PromptLibrary:
    """可复用的提示词模板库"""
    TRANSLATOR = ChatPromptTemplate.from_messages([
        ("system", "你是专业翻译，精通{source_lang}和{target_lang}"),
        ("user", "翻译以下文本：\n{text}")
    ])

    CODE_REVIEWER = ChatPromptTemplate.from_messages([
        ("system", "你是{language}代码审查专家，重点关注{focus}"),
        ("user", "审查代码：\n```{language}\n{code}\n```")
    ])

    SUMMARIZER = ChatPromptTemplate.from_messages([
        ("system", "你是内容摘要专家"),
        ("user", "将以下内容总结为{num}个要点：\n{content}")
    ])

    TUTOR = ChatPromptTemplate.from_messages([
        ("system", "你是{subject}导师，学生水平：{level}"),
        ("user", "{question}")
    ])

其它文件中进行调用：

In [30]:
# from templates import PromptLibrary

messages = PromptLibrary.TRANSLATOR.format_messages(
    source_lang="英语",
    target_lang="中文",
    text="Hello World"
)

print(messages)

[SystemMessage(content='你是专业翻译，精通英语和中文', additional_kwargs={}, response_metadata={}), HumanMessage(content='翻译以下文本：\nHello World', additional_kwargs={}, response_metadata={})]


举例2：

In [ ]:
# templates/
# ├── __init__.py
# ├── common.py         # 通用模板
# ├── translation.py    # 翻译相关
# └── coding.py         # 编程相关

from langchain_core.prompts import ChatPromptTemplate

# common.py
FRIENDLY_ASSISTANT = ChatPromptTemplate.from_messages([
    ("system", "你是一个友好的助手"),
    ("user", "{input}")
])

# translation.py
TRANSLATOR = ChatPromptTemplate.from_messages([
        ("system", "你是专业翻译，精通{source_lang}和{target_lang}"),
        ("user", "翻译以下文本：\n{text}")
])

# coding.py
CODE_REVIEWER = ChatPromptTemplate.from_messages([
        ("system", "你是{language}代码审查专家，重点关注{focus}"),
        ("user", "审查代码：\n```{language}\n{code}\n```")
])

## 4、模板组合

方法 1：字符串组合

In [33]:
# 定义可复用的部分
role_part = "你是一个{domain}专家。"
style_part = "回答风格：{style}。"
constraint_part = "限制：{constraint}。"
# 组合
full_system = role_part + style_part + constraint_part
print(full_system)
print("\n")
template = ChatPromptTemplate.from_messages([
    ("system", full_system),
    ("user", "{question}")
])
print(template)

你是一个{domain}专家。回答风格：{style}。限制：{constraint}。


input_variables=['constraint', 'domain', 'question', 'style'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['constraint', 'domain', 'style'], input_types={}, partial_variables={}, template='你是一个{domain}专家。回答风格：{style}。限制：{constraint}。'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})]


方法 2：使用 + 运算符

In [34]:
template1 = ChatPromptTemplate.from_messages([
    ("system", "你是助手")
])
template2 = ChatPromptTemplate.from_messages([
    ("user", "{input}")
])
# 组合（LangChain 1.0 支持）
combined = template1 + template2
print(combined)

input_variables=['input'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='你是助手'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})]
